In [1]:
import csv

# Fichiers d'entrée et de sortie
INPUT_FILE = "Accomplissements Progarmor - Feuille 1 (4).csv"
OUTPUT_FILE = "Fichier 2 Complet.csv"

# Les colonnes de niveaux dans le fichier 1
LEVEL_COLS = [
    "0 - Bois - Mode canapé", 
    "1 - Pierre - Débutant", 
    "2 - Fonte - Pratiquant", 
    "3 - Acier - Confirmé", 
    "4 - Titane - Athlète", 
    "5 - Diamant - Elite"
]

# Les en-têtes du fichier 2
OUTPUT_HEADERS = [
    "possible", "type", "level", "name_fr", "name_en", "description_fr", "description_en", 
    "hint_fr", "hint_en", "picture", "condition_code", "collection", "field", "howMany", 
    "datetimeBetweenStart", "datetimeBetweenEnd", "timeBetweenStart", "timeBetweenEnd", 
    "weightLoadMin", "valueMin", "variationIds", "variationType", "conditionValue"
]

# Dictionnaire pour mapper les descriptions dynamiques et les champs techniques
CONDITION_MAP = {
    "réaliser X séances": {
        "desc_fr": "Réalise {X} séances", "desc_en": "Complete {X} workouts",
        "tech": {"condition_code": "count", "collection": "seances", "howMany": "{X}"}
    },
    "faire X séances de plus de 1h30": {
        "desc_fr": "Fais {X} séances de plus de 1h30", "desc_en": "Complete {X} workouts longer than 1h30",
        "tech": {"condition_code": "count", "collection": "seances", "howMany": "{X}"}
    },
    "faire X séances sans aucun muscle bas du corps": {
        "desc_fr": "Fais {X} séances sans solliciter le bas du corps", "desc_en": "Complete {X} upper body only workouts",
        "tech": {"condition_code": "count", "collection": "seances", "howMany": "{X}"}
    },
    "faire X séances de moins de 30 minutes": {
        "desc_fr": "Fais {X} séances de moins de 30 minutes", "desc_en": "Complete {X} workouts under 30 minutes",
        "tech": {"condition_code": "count", "collection": "seances", "howMany": "{X}"}
    },
    "faire X séances entre 22h et 8h du matin": {
        "desc_fr": "Fais {X} séances entre 22h et 8h", "desc_en": "Complete {X} night owl workouts",
        "tech": {"condition_code": "count", "collection": "seances", "howMany": "{X}", "timeBetweenStart": "22:00", "timeBetweenEnd": "08:00"}
    },
    "faire X séances avec au moins 2 exos street (variantes comprises)": {
        "desc_fr": "Fais {X} séances avec au moins 2 exos de street workout", "desc_en": "Complete {X} workouts with at least 2 street workout exercises",
        "tech": {"condition_code": "count", "collection": "seances", "howMany": "{X}"}
    },
    "kgs soulevés total": {
        "desc_fr": "Soulève un total de {X} kgs", "desc_en": "Lift a total of {X} kgs",
        "tech": {"condition_code": "volume_total", "collection": "seancesets", "valueMin": "{X}"}
    },
    "faire une séance le jour d'anniversaire": {
        "desc_fr": "Fais une séance le jour de ton anniversaire", "desc_en": "Workout on your birthday",
        "tech": {"condition_code": "birthday_workout"}
    },
    "nombres de prs totales": {
        "desc_fr": "Bats {X} records personnels (PRs)", "desc_en": "Beat {X} personal records (PRs)",
        "tech": {"condition_code": "count_prs", "howMany": "{X}"}
    },
    "Meilleure série de semaines consécutives avec séance": {
        "desc_fr": "Atteins une série de {X} semaines avec séance", "desc_en": "Achieve a {X} week workout streak",
        "tech": {"condition_code": "streak_weeks", "howMany": "{X}"}
    },
    "pr squat (n'importe quelle variante) en kgs": {
        "desc_fr": "Atteins un PR de {X}kg au squat (toutes variantes)", "desc_en": "Reach a squat PR of {X}kg (any variation)",
        "tech": {"condition_code": "exercise_pr_weight", "collection": "seancesets", "field": "weightLoad", "valueMin": "{X}", "variationIds": "669ced7e665a3ffe77714374", "conditionValue": "squat_family"}
    },
    "pr dc (n'importe quelle variante) en kgs": {
        "desc_fr": "Atteins un PR de {X}kg au développé couché (toutes variantes)", "desc_en": "Reach a bench press PR of {X}kg (any variation)",
        "tech": {"condition_code": "exercise_pr_weight", "collection": "seancesets", "field": "weightLoad", "valueMin": "{X}", "conditionValue": "bench_press_family"}
    },
    "pr sdt (n'importe quelle variante) en kgs": {
        "desc_fr": "Atteins un PR de {X}kg au soulevé de terre (toutes variantes)", "desc_en": "Reach a deadlift PR of {X}kg (any variation)",
        "tech": {"condition_code": "exercise_pr_weight", "collection": "seancesets", "field": "weightLoad", "valueMin": "{X}", "conditionValue": "deadlift_family"}
    },
    "pr hip thrust (n'importe quelle variante) en kgs": {
        "desc_fr": "Atteins un PR de {X}kg au hip thrust (toutes variantes)", "desc_en": "Reach a hip thrust PR of {X}kg (any variation)",
        "tech": {"condition_code": "exercise_pr_weight", "collection": "seancesets", "field": "weightLoad", "valueMin": "{X}", "conditionValue": "hip_thrust_family"}
    },
    "pr tractions (n'importe quelle variante) en kgs": {
        "desc_fr": "Atteins un PR de {X}kg aux tractions (toutes variantes)", "desc_en": "Reach a pull-up PR of {X}kg (any variation)",
        "tech": {"condition_code": "exercise_pr_weight", "collection": "seancesets", "field": "weightLoad", "valueMin": "{X}", "conditionValue": "pullup_family"}
    },
    "pr dips (n'importe quelle variante) en kgs": {
        "desc_fr": "Atteins un PR de {X}kg aux dips (toutes variantes)", "desc_en": "Reach a dips PR of {X}kg (any variation)",
        "tech": {"condition_code": "exercise_pr_weight", "collection": "seancesets", "field": "weightLoad", "valueMin": "{X}", "conditionValue": "dips_family"}
    }
}

with open(INPUT_FILE, mode='r', encoding='utf-8-sig') as infile, \
     open(OUTPUT_FILE, mode='w', encoding='utf-8', newline='') as outfile:
    
    reader = csv.DictReader(infile)
    writer = csv.DictWriter(outfile, fieldnames=OUTPUT_HEADERS)
    writer.writeheader()
    
    for row in reader:
        condition_brute = row.get('condition', '').strip()
        map_info = CONDITION_MAP.get(condition_brute, {})
        
        # Parcourir chaque niveau (0 à 5)
        for level_idx, level_col_name in enumerate(LEVEL_COLS):
            valeur_niveau = row.get(level_col_name, '').strip()
            
            # Si pas de valeur requise pour ce niveau, on passe
            if not valeur_niveau:
                continue
            
            # Préparer la nouvelle ligne
            new_row = {col: "" for col in OUTPUT_HEADERS}
            new_row['possible'] = row.get('possible', 'TRUE')
            new_row['type'] = row.get('type', '')
            new_row['level'] = str(level_idx)
            new_row['name_fr'] = row.get('nom fr', '')
            new_row['name_en'] = row.get('nom en', '')
            
            # Injection des descriptions
            if map_info:
                new_row['description_fr'] = map_info["desc_fr"].replace("{X}", valeur_niveau)
                new_row['description_en'] = map_info["desc_en"].replace("{X}", valeur_niveau)
                
                # Injection des valeurs techniques
                for tech_key, tech_val in map_info.get("tech", {}).items():
                    new_row[tech_key] = tech_val.replace("{X}", valeur_niveau)
            else:
                # Fallback si la condition n'est pas dans le dictionnaire
                new_row['description_fr'] = f"{condition_brute} ({valeur_niveau})"
                new_row['description_en'] = f"{condition_brute} ({valeur_niveau})"

            writer.writerow(new_row)

print(f"Transformation terminée. Résultat enregistré dans {OUTPUT_FILE}")

Transformation terminée. Résultat enregistré dans Fichier 2 Complet.csv
